# Feature Extraction (Spoken Language Identification)

**Goal:** Extract fixed-length audio features from 4 languages (`de`, `es`, `it`, `ko`) for use in **Classification** and **Clustering**.

**Inputs**
- `metadata.csv` containing file paths and labels
- Cleaned audio files under `data/clean/...`

**Outputs**
- `features.csv`
- If any file fails: `bad_files*.csv`

## 4) Feature extraction

This is the main feature-extraction code (MFCC + Δ/ΔΔ + spectral/energy features + duration).  
At the end, the output is saved as `features.csv`.

In [ ]:
import os
import numpy as np
import pandas as pd
import librosa

# ---------------------------
# 1) Load metadata + sanity check
# ---------------------------
meta = pd.read_csv("metadata.csv")

# Ensure we are using CLEAN audio (recommended)
if not meta["filepath"].astype(str).str.contains("data/clean").all():
    print(" Warning: Some filepaths are not from data/clean. Please check metadata.csv.")

# ---------------------------
# 2) Feature extraction function
# ---------------------------
def stats_2(x: np.ndarray):
    """Return mean and std of a 1D array."""
    return np.mean(x), np.std(x)

def feat_stats(mat: np.ndarray):
    """Return per-row mean and std for a 2D matrix [n_features, n_frames]."""
    return mat.mean(axis=1), mat.std(axis=1)

def extract_features_v2(path, sr=16000, n_mfcc=20):
    """
    Professional baseline features for speech/language:
    - MFCC (mean/std)
    - MFCC delta (mean/std)
    - MFCC delta-delta (mean/std)
    - ZCR (mean/std)
    - Spectral centroid (mean/std)
    - Spectral rolloff (mean/std)
    - Spectral bandwidth (mean/std)
    - Spectral flatness (mean/std)
    - RMS energy (mean/std)
    Also returns duration seconds for reporting.
    """
    y, _sr = librosa.load(path, sr=sr, mono=True)
    duration_sec = len(y) / sr

    # MFCC base
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    mfcc_mean, mfcc_std = feat_stats(mfcc)

    # Delta and Delta-Delta
    mfcc_d1 = librosa.feature.delta(mfcc)
    mfcc_d2 = librosa.feature.delta(mfcc, order=2)
    d1_mean, d1_std = feat_stats(mfcc_d1)
    d2_mean, d2_std = feat_stats(mfcc_d2)

    # Time/Frequency features
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    rolloff  = librosa.feature.spectral_rolloff(y=y, sr=sr)[0]
    bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    flatness  = librosa.feature.spectral_flatness(y=y)[0]
    rms = librosa.feature.rms(y=y)[0]

    zcr_m, zcr_s = stats_2(zcr)
    cen_m, cen_s = stats_2(centroid)
    rol_m, rol_s = stats_2(rolloff)
    bw_m,  bw_s  = stats_2(bandwidth)
    fl_m,  fl_s  = stats_2(flatness)
    rms_m, rms_s = stats_2(rms)

    feats = np.hstack([
        mfcc_mean, mfcc_std,
        d1_mean, d1_std,
        d2_mean, d2_std,
        zcr_m, zcr_s,
        cen_m, cen_s,
        rol_m, rol_s,
        bw_m,  bw_s,
        fl_m,  fl_s,
        rms_m, rms_s
    ]).astype(np.float32)

    return feats, duration_sec

# ---------------------------
# 3) Run extraction (robust)
# ---------------------------
X = []
durations = []
bad_files = []

expected_len = None

for i, fp in enumerate(meta["filepath"], start=1):
    fp2 = fp.replace("/", os.sep)

    try:
        feats, dur = extract_features_v2(fp2, sr=16000, n_mfcc=20)
        if expected_len is None:
            expected_len = len(feats)
        X.append(feats)
        durations.append(dur)
    except Exception as e:
        bad_files.append((fp, str(e)))
        if expected_len is None:
            # fallback expected length for v2 (only if first file fails)
            # length = MFCC(20)*2 + d1(20)*2 + d2(20)*2 + 9 features *2? (zcr,cen,rol,bw,flat,rms = 6 features => 12)
            # actually: zcr/cen/rol/bw/flat/rms = 6 -> 12 numbers
            # total = 20*2 + 20*2 + 20*2 + 20*2 + 20*2 + 20*2? no
            # We won't hardcode; just set to 0 and fill later.
            expected_len = 0
        X.append(np.full(expected_len, np.nan, dtype=np.float32))
        durations.append(np.nan)

    if i % 50 == 0:
        print(f"Processed {i}/{len(meta)}")

X = np.vstack(X)

# ---------------------------
# 4) Build feature columns
# ---------------------------
feat_cols = []

# MFCC base
feat_cols += [f"mfcc{i}_mean" for i in range(1, 21)]
feat_cols += [f"mfcc{i}_std"  for i in range(1, 21)]

# Delta
feat_cols += [f"mfcc_d1_{i}_mean" for i in range(1, 21)]
feat_cols += [f"mfcc_d1_{i}_std"  for i in range(1, 21)]

# Delta-Delta
feat_cols += [f"mfcc_d2_{i}_mean" for i in range(1, 21)]
feat_cols += [f"mfcc_d2_{i}_std"  for i in range(1, 21)]

# Other features (mean/std)
feat_cols += [
    "zcr_mean","zcr_std",
    "centroid_mean","centroid_std",
    "rolloff_mean","rolloff_std",
    "bandwidth_mean","bandwidth_std",
    "flatness_mean","flatness_std",
    "rms_mean","rms_std"
]

# Safety check
assert X.shape[1] == len(feat_cols), f"Feature length mismatch: X has {X.shape[1]} cols but feat_cols has {len(feat_cols)}"

# ---------------------------
# 5) Save features.csv + reports
# ---------------------------
df = pd.concat([meta.reset_index(drop=True),
                   pd.DataFrame(X, columns=feat_cols),
                   pd.Series(durations, name="duration_sec")], axis=1)

before = len(df)
df_clean = df.dropna().reset_index(drop=True)
after = len(df_clean)

df_clean.to_csv("features.csv", index=False)

print("Saved features.csv with shape:", df_clean.shape)
print(f"Dropped {before-after} rows due to errors.")

# Drop analysis by language (useful in report)
if before - after > 0:
    dropped = df[df.isna().any(axis=1)]
    print("\nDropped by language:")
    print(dropped["lang"].value_counts())

# Bad files log
if bad_files:
    pd.DataFrame(bad_files, columns=["filepath","error"]).to_csv("bad_files.csv", index=False)
    print("Saved bad_files.csv for debugging.")

# Duration summary (useful for report)
print("\nDuration summary (sec):")
print(df_clean["duration_sec"].describe())


Processed 50/720
Processed 100/720
Processed 150/720
Processed 200/720
Processed 250/720
Processed 300/720
Processed 350/720
Processed 400/720
Processed 450/720
Processed 500/720
Processed 550/720
Processed 600/720
Processed 650/720
Processed 700/720

✅ Saved features_v2.csv with shape: (720, 136)
Dropped 0 rows due to errors.

Duration summary (sec):
count     720.000000
mean       63.185709
std        55.216793
min        39.000813
25%        58.000000
50%        60.684687
75%        63.907328
max      1534.249812
Name: duration_sec, dtype: float64


## 5) Validate the output

A small sanity check:
- dataframe shape
- NaN count (preferably 0)

In [ ]:
df = pd.read_csv("features.csv")
print(df.shape)
print(df.isna().sum().sum())

(720, 136)
0
